# 02 - Data Ingestion: PDFs → Weaviate

Drop PDFs in `data/pdfs/`:
| File | URL |
|---|---|
| llm_supply_chain.pdf | https://arxiv.org/abs/2307.03875 |
| graph_digital_twin_scm.pdf | https://arxiv.org/abs/2504.03692 |
| llm_network_optimization.pdf | https://arxiv.org/abs/2508.21622 |
| tutorialspoint_scm.pdf | already have |

In [1]:
import os, hashlib
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

HF_API_KEY       = os.getenv('HF_API_KEY')
COHERE_API_KEY   = os.getenv('COHERE_API_KEY', '')
WEAVIATE_URL     = os.getenv('WEAVIATE_URL', '').rstrip('/')
WEAVIATE_API_KEY = os.getenv('WEAVIATE_API_KEY')
COLLECTION       = 'ScmChunks'

PDF_DIR       = Path('../data/pdfs')
CHUNK_SIZE    = 600
CHUNK_OVERLAP = 80
BATCH_SIZE    = 50

pdfs = list(PDF_DIR.glob('*.pdf'))
print(f'Found {len(pdfs)} PDFs:')
for p in pdfs: print(f'  {p.name}  ({p.stat().st_size//1024:,} KB)')
if not pdfs: raise FileNotFoundError(f'No PDFs in {PDF_DIR}')

Found 4 PDFs:
  2307.03875v2.pdf  (1,034 KB)
  2504.03692v1.pdf  (2,175 KB)
  2508.21622v1.pdf  (958 KB)
  supply_chain_management_tutorial.pdf  (1,421 KB)


## 1. Chunk PDFs

In [2]:
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_pdf(path):
    doc  = fitz.open(str(path))
    text = '\n'.join(page.get_text() for page in doc)
    spl  = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    return [{'chunk_id': hashlib.md5(f'{path.name}_{i}'.encode()).hexdigest(),
              'source_file': path.name, 'chunk_idx': i, 'text': t, 'char_count': len(t)}
            for i, t in enumerate(spl.split_text(text))]

all_chunks = []
for pdf in pdfs:
    chunks = chunk_pdf(pdf)
    all_chunks.extend(chunks)
    print(f'  {pdf.name}: {len(chunks)} chunks')
print(f'Total: {len(all_chunks):,}')

  2307.03875v2.pdf: 137 chunks
  2504.03692v1.pdf: 332 chunks
  2508.21622v1.pdf: 99 chunks
  supply_chain_management_tutorial.pdf: 201 chunks
Total: 769


## 2. Insert into Weaviate

> **Idempotent**: skips insertion if collection already has data.
> To re-ingest, run notebook 01 first to drop & recreate the collection.

In [3]:
import weaviate
from weaviate.classes.init import Auth, AdditionalConfig, Timeout

_headers = {}
if HF_API_KEY:
    _headers['X-HuggingFace-Api-Key'] = HF_API_KEY.strip()
if COHERE_API_KEY:
    _headers['X-Cohere-Api-Key'] = COHERE_API_KEY.strip()

client = weaviate.connect_to_weaviate_cloud(
    cluster_url      = WEAVIATE_URL,
    auth_credentials = Auth.api_key(WEAVIATE_API_KEY),
    headers          = _headers,
    additional_config= AdditionalConfig(timeout=Timeout(init=60, query=30, insert=120)),
    skip_init_checks = True,
)
collection    = client.collections.get(COLLECTION)
existing_count= collection.aggregate.over_all(total_count=True).total_count
print(f'Existing objects: {existing_count}')

if existing_count > 0:
    print(f'Skipping — collection already has {existing_count} objects.')
    print('To re-ingest: re-run notebook 01 to drop & recreate the collection first.')
else:
    inserted = 0
    with collection.batch.fixed_size(batch_size=BATCH_SIZE) as batch:
        for chunk in all_chunks:
            batch.add_object(properties=chunk)
            inserted += 1
            if inserted % 100 == 0: print(f'  {inserted}/{len(all_chunks)}')
    print(f'Inserted {inserted} chunks')

Existing objects: 0
  100/769
  200/769
  300/769
  400/769
  500/769
  600/769
  700/769
Inserted 769 chunks


## 3. Validate

In [4]:
results = collection.query.near_text(
    query='supply chain disruption risk mitigation', limit=3,
    return_properties=['source_file', 'chunk_idx', 'text'],
)
for o in results.objects:
    p = o.properties
    print(f'[{p["source_file"]} chunk#{p["chunk_idx"]}]')
    print(f'  {p["text"][:150]}...\n')

print(f'Total: {collection.aggregate.over_all(total_count=True).total_count}')
client.close()

[2504.03692v1.pdf chunk#156]
  long-term consequences, such as shifts in market demand due to a supply chain disruption [157]. By
running “what-if” simulations, we can test differen...

[2504.03692v1.pdf chunk#153]
  Simulation (DES), which models supply chain operations as a series of distinct events occurring over time.
This method helps us analyze disruptions, s...

[2504.03692v1.pdf chunk#76]
  disruption in one part of the system—whether due to a natural disaster, geopolitical tensions, or supply
chain inefficiencies—can have a cascading eff...

Total: 769
